In [ ]:
"""
================================================================================
CRESTA MLE INTERVIEW — PRACTICAL CODING PROBLEM
================================================================================
DOMAIN:  Call Transcript Summarization with Structured Output
TIME:    90 minutes
LEVEL:   Mid-Senior MLE

SCENARIO
--------
You are an MLE on the Agent Assist team at Cresta. After every customer call,
agents must write a summary in the CRM — this "after-call work" (ACW) adds
30-90 seconds per call. At 50,000 calls/day, that's 400-750 hours of agent
time per day spent typing summaries.

Cresta's AI Summaries product generates structured summaries INSTANTLY at call
end, reducing ACW to a quick review + click "accept." The summary must be:

  1. STRUCTURED — a fixed JSON schema, not free-form text
  2. FAITHFUL — every claim must be grounded in the transcript (no hallucination)
  3. ACTIONABLE — follow-up items must be specific and assignable
  4. FAST — generated within 3 seconds of call end

The structured schema your summaries must follow:
{
    "reason_for_call": str,        # 1-2 sentences: why did the customer call?
    "resolution": str,             # 1-2 sentences: how was the issue resolved?
    "resolution_status": str,      # one of: "resolved", "escalated", "pending", "unresolved"
    "follow_up_actions": [str],    # list of specific next steps (can be empty)
    "customer_sentiment": str,     # one of: "positive", "neutral", "frustrated", "angry"
    "key_entities": {              # extracted structured data
        "account_id": str | null,
        "product_mentioned": str | null,
        "dollar_amount": str | null,
    }
}

YOUR TASK  (4 parts, increasing difficulty)
-------------------------------------------
Part 1: Transcript parsing & preprocessing            [15 min]
Part 2: Prompt engineering for structured extraction   [25 min]
Part 3: Faithfulness evaluation (hallucination check)  [25 min]
Part 4: Evaluation framework + production design       [25 min]

Each part has a function stub. Fill them in. Run this file to verify.

NOTE: Parts 2-4 are designed to work WITHOUT an actual LLM API call.
      You'll write the prompt construction, parsing, and evaluation logic.
      Where an LLM call would go, use the provided mock_llm_call() function.
================================================================================
"""

import json
import re
import hashlib
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass, field, asdict
from collections import Counter
import difflib


# ─────────────────────────────────────────────────────────────────────────────
# SYNTHETIC TRANSCRIPT GENERATOR  (do NOT modify)
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class Utterance:
    role: str           # "agent" or "customer"
    text: str
    timestamp_sec: int  # seconds from call start


@dataclass
class Transcript:
    call_id: str
    utterances: List[Utterance]
    metadata: Dict[str, Any] = field(default_factory=dict)

    def to_text(self) -> str:
        """Render as plain text dialogue."""
        lines = []
        for u in self.utterances:
            mins, secs = divmod(u.timestamp_sec, 60)
            lines.append(f"[{mins:02d}:{secs:02d}] {u.role.upper()}: {u.text}")
        return "\n".join(lines)

    def total_duration_sec(self) -> int:
        return self.utterances[-1].timestamp_sec if self.utterances else 0

    def word_count(self) -> int:
        return sum(len(u.text.split()) for u in self.utterances)


# --- Transcript library: 6 realistic call scenarios ---

def _make_billing_dispute() -> Tuple[Transcript, Dict]:
    """Billing dispute — resolved with credit."""
    utts = [
        Utterance("agent", "Thank you for calling Meridian Wireless, my name is Sarah. How can I help you today?", 0),
        Utterance("customer", "Hi Sarah, I'm calling because I was charged $47.99 for something called a premium data add-on, but I never signed up for that.", 5),
        Utterance("agent", "I'm sorry to hear about that unexpected charge. Let me pull up your account. Can I get your account number?", 18),
        Utterance("customer", "Sure, it's AC-7842961.", 28),
        Utterance("agent", "Thank you. I can see your account now. I do see a premium data add-on was added on March 15th. Let me look into how that happened.", 35),
        Utterance("customer", "I definitely didn't add that myself. I've had the same basic plan for two years.", 52),
        Utterance("agent", "I understand your frustration. It looks like this may have been added during a system migration we did last month. I can see several customers were affected. I'm going to remove that add-on right now and issue a full refund of $47.99 to your next billing cycle.", 60),
        Utterance("customer", "Okay, that works. Will it show up on my next statement?", 85),
        Utterance("agent", "Yes, you'll see a credit of $47.99 on your next statement. I've also added a note to your account so this won't happen again. Is there anything else I can help with?", 92),
        Utterance("customer", "No, that's all. Thanks for sorting it out quickly.", 110),
        Utterance("agent", "You're welcome! Thank you for being a loyal customer for two years. Have a great day.", 118),
    ]
    ground_truth = {
        "reason_for_call": "Customer was charged $47.99 for a premium data add-on they did not sign up for.",
        "resolution": "Agent identified the charge was caused by a system migration error, removed the add-on, and issued a full $47.99 refund to the next billing cycle.",
        "resolution_status": "resolved",
        "follow_up_actions": [],
        "customer_sentiment": "neutral",
        "key_entities": {
            "account_id": "AC-7842961",
            "product_mentioned": "premium data add-on",
            "dollar_amount": "$47.99",
        },
    }
    return Transcript("call_001", utts, {"team": "billing"}), ground_truth


def _make_tech_escalation() -> Tuple[Transcript, Dict]:
    """Technical issue — escalated to tier 2."""
    utts = [
        Utterance("agent", "Meridian Wireless support, this is James. What can I help you with?", 0),
        Utterance("customer", "My internet has been dropping out every 20 minutes for the past three days. I work from home and this is seriously affecting my job.", 6),
        Utterance("agent", "I'm very sorry to hear that, especially with you working from home. Let me run a diagnostic on your connection. Can I have your account number?", 22),
        Utterance("customer", "It's AC-3391054.", 35),
        Utterance("agent", "Thank you. I'm running a line test now... I can see there are significant packet loss events on your connection. Let me check if there are any known outages in your area.", 42),
        Utterance("customer", "I already checked online and it says no outages, but clearly something is wrong.", 68),
        Utterance("agent", "You're right, there's no area-wide outage, but I'm seeing some signal degradation that suggests a possible line issue between your home and our node. Unfortunately, this is beyond what I can fix remotely.", 78),
        Utterance("customer", "So what happens now? I really can't keep dealing with this.", 100),
        Utterance("agent", "I completely understand the urgency. I'm going to escalate this to our Tier 2 network engineering team. They'll need to dispatch a technician to inspect the line. I'm requesting a priority appointment for you — someone should contact you within 24 hours to schedule a visit.", 108),
        Utterance("customer", "Twenty-four hours? Is there any way to speed that up?", 135),
        Utterance("agent", "I've flagged it as urgent since you work from home. In the meantime, I'd recommend connecting directly to your modem with an ethernet cable if possible — that may reduce the drops. I'm also applying a $25 service credit to your account for the inconvenience.", 142),
        Utterance("customer", "Fine. I'll try the ethernet cable. Just make sure someone actually calls me.", 170),
        Utterance("agent", "Absolutely. You'll receive a confirmation email shortly with your ticket number TK-88421. Is there anything else?", 180),
        Utterance("customer", "No, that's it.", 195),
        Utterance("agent", "Thank you for your patience. We'll get this resolved for you. Have a good day.", 200),
    ]
    ground_truth = {
        "reason_for_call": "Customer's internet has been dropping out every 20 minutes for three days, affecting their ability to work from home.",
        "resolution": "Agent ran diagnostics and found signal degradation indicating a line issue. Escalated to Tier 2 network engineering for a technician dispatch with priority scheduling. Applied a $25 service credit.",
        "resolution_status": "escalated",
        "follow_up_actions": [
            "Tier 2 network engineering to contact customer within 24 hours to schedule technician visit",
            "Confirmation email with ticket TK-88421 to be sent to customer",
        ],
        "customer_sentiment": "frustrated",
        "key_entities": {
            "account_id": "AC-3391054",
            "product_mentioned": None,
            "dollar_amount": "$25",
        },
    }
    return Transcript("call_002", utts, {"team": "technical"}), ground_truth


def _make_cancellation_save() -> Tuple[Transcript, Dict]:
    """Cancellation attempt — saved with retention offer."""
    utts = [
        Utterance("agent", "Thank you for calling Meridian Wireless, I'm Lisa. How can I assist you?", 0),
        Utterance("customer", "I want to cancel my account. I'm switching to Apex Mobile.", 7),
        Utterance("agent", "I'm sorry to hear you're considering leaving us. May I ask what's prompting the switch?", 15),
        Utterance("customer", "Honestly, your prices keep going up. I'm paying $89 a month and Apex is offering the same thing for $59.", 24),
        Utterance("agent", "I appreciate you being upfront about that. You've been with us for three years and I want to make sure we explore all options before you go. Let me see what I can do for your account AC-5567823.", 38),
        Utterance("customer", "I've already made up my mind, but go ahead.", 58),
        Utterance("agent", "I understand. Looking at your usage, I see you're on our Unlimited Plus plan. We actually have a loyalty rate that I can apply — it would bring your monthly cost down to $62 per month, and I can lock that rate in for 12 months. You'd also keep your current device installment plan.", 65),
        Utterance("customer", "$62? That's pretty close to what Apex offered. What's the catch?", 92),
        Utterance("agent", "No catch — it's a 12-month price lock for customers who've been with us over two years. The only thing to note is that if you cancel within those 12 months, the remaining balance on your device installment would come due.", 100),
        Utterance("customer", "Okay, let me think... yeah, I'll take that deal. $62 locked in for a year works for me.", 120),
        Utterance("agent", "Wonderful! I've applied the loyalty rate to your account effective today. You'll see the new rate on your next bill. I'm also going to send you a confirmation email. Is there anything else I can help with?", 135),
        Utterance("customer", "No, that's it. Thanks Lisa, I almost left but this is a good deal.", 155),
        Utterance("agent", "I'm glad we could work it out! Thank you for staying with Meridian. Have a great evening.", 163),
    ]
    ground_truth = {
        "reason_for_call": "Customer called to cancel their account and switch to a competitor (Apex Mobile) due to pricing concerns — they were paying $89/month.",
        "resolution": "Agent offered a loyalty rate of $62/month locked in for 12 months on the Unlimited Plus plan. Customer accepted and decided to stay.",
        "resolution_status": "resolved",
        "follow_up_actions": [
            "Confirmation email to be sent to customer with new rate details",
        ],
        "customer_sentiment": "positive",
        "key_entities": {
            "account_id": "AC-5567823",
            "product_mentioned": "Unlimited Plus plan",
            "dollar_amount": "$62",
        },
    }
    return Transcript("call_003", utts, {"team": "retention"}), ground_truth


def _make_complex_multi_issue() -> Tuple[Transcript, Dict]:
    """Complex: multiple issues in one call — partial resolution."""
    utts = [
        Utterance("agent", "Meridian Wireless, this is David. How can I help?", 0),
        Utterance("customer", "Hi, I have a couple things. First, I need to update my address because I'm moving next week. Second, my last bill has a charge I don't recognize.", 5),
        Utterance("agent", "Sure, I can help with both. Let me start with the address change. What's your account number?", 22),
        Utterance("customer", "AC-9912347. My new address is 445 Birch Lane, Apartment 3B, Portland, OR 97201.", 30),
        Utterance("agent", "Got it. I've updated your service address to 445 Birch Lane, Apt 3B, Portland, OR 97201. The change takes effect immediately for billing, but if you have home internet with us, we'll need to schedule a transfer. Do you have our home internet?", 48),
        Utterance("customer", "Yes, I have the 500 Mbps plan.", 72),
        Utterance("agent", "Okay, the internet transfer will need to be scheduled separately. I'll put in a transfer request — our installations team will reach out within 48 hours to set a date. Now, regarding the unrecognized charge — can you tell me more about it?", 78),
        Utterance("customer", "There's a $14.99 charge labeled 'Device Protection Plus' on my April statement. I never added any device protection.", 102),
        Utterance("agent", "Let me look into that... I can see Device Protection Plus was added to your account on February 2nd. It looks like it was added through our mobile app. Do you recall making any changes around that time?", 115),
        Utterance("customer", "I might have been browsing the app but I definitely didn't mean to add anything. Can you just remove it?", 140),
        Utterance("agent", "I've removed Device Protection Plus from your account right now. However, since it shows as added through the app, I can only issue a one-month refund of $14.99. For the prior months, I'll need to submit a dispute form to our billing review team. You should hear back within 5-7 business days.", 150),
        Utterance("customer", "That's annoying but fine. Just make sure I get my money back for all three months.", 180),
        Utterance("agent", "I completely understand. I've submitted the dispute for the remaining $29.98 covering February and March. Your reference number is RF-44217. To summarize everything: address updated, internet transfer request submitted, Device Protection Plus removed with $14.99 refunded now, and $29.98 dispute pending. Anything else?", 190),
        Utterance("customer", "No, I think that covers it. Thanks David.", 225),
        Utterance("agent", "You're welcome. Good luck with the move! Have a great day.", 232),
    ]
    ground_truth = {
        "reason_for_call": "Customer called with two issues: (1) needed to update their address due to an upcoming move, and (2) had an unrecognized $14.99 charge for Device Protection Plus on their bill.",
        "resolution": "Address updated to 445 Birch Lane, Apt 3B, Portland, OR 97201. Internet transfer request submitted. Device Protection Plus removed with $14.99 refunded immediately; dispute for remaining $29.98 (Feb-Mar) submitted to billing review.",
        "resolution_status": "pending",
        "follow_up_actions": [
            "Installations team to contact customer within 48 hours to schedule internet transfer",
            "Billing review team to process $29.98 dispute (ref RF-44217) within 5-7 business days",
        ],
        "customer_sentiment": "neutral",
        "key_entities": {
            "account_id": "AC-9912347",
            "product_mentioned": "Device Protection Plus",
            "dollar_amount": "$14.99",
        },
    }
    return Transcript("call_004", utts, {"team": "billing"}), ground_truth


def _make_short_simple() -> Tuple[Transcript, Dict]:
    """Short, simple inquiry — resolved in under a minute."""
    utts = [
        Utterance("agent", "Meridian Wireless, how can I help?", 0),
        Utterance("customer", "Quick question — when does my contract end? I'm on account AC-2210038.", 4),
        Utterance("agent", "Let me check... your current 24-month agreement ends on August 15th, 2026. After that, you'll automatically switch to month-to-month.", 12),
        Utterance("customer", "Perfect, that's all I needed. Thanks!", 28),
        Utterance("agent", "Happy to help. Have a good one!", 33),
    ]
    ground_truth = {
        "reason_for_call": "Customer inquired about their contract end date.",
        "resolution": "Agent confirmed the 24-month contract ends on August 15th, 2026, after which service converts to month-to-month.",
        "resolution_status": "resolved",
        "follow_up_actions": [],
        "customer_sentiment": "positive",
        "key_entities": {
            "account_id": "AC-2210038",
            "product_mentioned": None,
            "dollar_amount": None,
        },
    }
    return Transcript("call_005", utts, {"team": "general"}), ground_truth


def _make_angry_unresolved() -> Tuple[Transcript, Dict]:
    """Angry customer — issue not resolved, threatens to leave."""
    utts = [
        Utterance("agent", "Thank you for calling Meridian Wireless, this is Tony. How can I help you today?", 0),
        Utterance("customer", "This is the THIRD time I'm calling about the same problem. Every single month my bill is wrong. I was told it would be $75 and it keeps coming in at $93.", 6),
        Utterance("agent", "I sincerely apologize for the repeated issue. Let me look into your account right away. Can I get your account number?", 25),
        Utterance("customer", "AC-6681200. And I want to talk to a supervisor this time. No offense but the last two agents promised to fix it and nothing changed.", 32),
        Utterance("agent", "I completely understand your frustration, and I don't blame you for wanting to speak with a supervisor. Before I transfer you, can I take a quick look to understand what's happening? I want to make sure the supervisor has all the context.", 48),
        Utterance("customer", "Fine, but make it quick. I've wasted enough time on this.", 68),
        Utterance("agent", "I see the issue — there's an $18 add-on for international calling that keeps getting reattached to your account. It was removed on March 3rd and March 28th, but it reappeared each time on the next billing cycle. This looks like it could be a system bug.", 75),
        Utterance("customer", "A system bug?! And nobody caught this in my last two calls?", 100),
        Utterance("agent", "I can see notes from the previous calls where the add-on was removed, but there's no record of anyone investigating why it keeps coming back. I'm going to flag this as a system issue and transfer you to my supervisor, Rachel, who can authorize a permanent fix and process the overcharge refunds. The total overpayment across three months would be $54.", 108),
        Utterance("customer", "Three months of overcharges. This is ridiculous. If this isn't fixed this time, I'm leaving.", 140),
        Utterance("agent", "I hear you and I'm documenting everything so you won't have to re-explain. I'm transferring you now to Rachel. Please stay on the line.", 150),
        Utterance("customer", "Alright.", 165),
    ]
    ground_truth = {
        "reason_for_call": "Customer called for the third time about recurring billing errors — they were promised a $75 monthly rate but are being charged $93 due to an international calling add-on that keeps reattaching after removal.",
        "resolution": "Agent identified the recurring $18 add-on as a likely system bug and transferred the customer to supervisor Rachel for a permanent fix and refund authorization.",
        "resolution_status": "escalated",
        "follow_up_actions": [
            "Supervisor Rachel to authorize permanent removal of international calling add-on and fix the system bug",
            "Process $54 refund for three months of overcharges",
        ],
        "customer_sentiment": "angry",
        "key_entities": {
            "account_id": "AC-6681200",
            "product_mentioned": "international calling add-on",
            "dollar_amount": "$54",
        },
    }
    return Transcript("call_006", utts, {"team": "billing"}), ground_truth


def get_all_transcripts() -> List[Tuple[Transcript, Dict]]:
    """Return all 6 transcripts with ground-truth summaries."""
    return [
        _make_billing_dispute(),
        _make_tech_escalation(),
        _make_cancellation_save(),
        _make_complex_multi_issue(),
        _make_short_simple(),
        _make_angry_unresolved(),
    ]


# ─────────────────────────────────────────────────────────────────────────────
# MOCK LLM FUNCTION  (simulates an API call — do NOT modify)
# ─────────────────────────────────────────────────────────────────────────────

def mock_llm_call(prompt: str, system_prompt: str = "") -> str:
    """
    Simulates an LLM API call. Returns a deterministic response based on the
    transcript content in the prompt. In a real implementation, this would be
    an API call to Claude, GPT-4, etc.

    You should write your code AS IF this were a real LLM call — structure your
    prompts properly, parse the output, handle errors. The mock just lets you
    test your pipeline without API keys.
    """
    # Detect which transcript is in the prompt and return a pre-built response
    if "AC-7842961" in prompt:
        return json.dumps({
            "reason_for_call": "Customer was charged $47.99 for a premium data add-on they did not authorize.",
            "resolution": "Agent removed the add-on and issued a full refund of $47.99 to the next billing cycle.",
            "resolution_status": "resolved",
            "follow_up_actions": [],
            "customer_sentiment": "neutral",
            "key_entities": {"account_id": "AC-7842961", "product_mentioned": "premium data add-on", "dollar_amount": "$47.99"},
        })
    elif "AC-3391054" in prompt:
        return json.dumps({
            "reason_for_call": "Customer reported internet dropping every 20 minutes for three days, impacting their remote work.",
            "resolution": "Diagnostics showed signal degradation from a line issue. Escalated to Tier 2 engineering with priority scheduling. $25 service credit applied.",
            "resolution_status": "escalated",
            "follow_up_actions": [
                "Tier 2 to contact customer within 24 hours for technician scheduling",
                "Send confirmation email with ticket TK-88421",
            ],
            "customer_sentiment": "frustrated",
            "key_entities": {"account_id": "AC-3391054", "product_mentioned": None, "dollar_amount": "$25"},
        })
    elif "AC-5567823" in prompt:
        return json.dumps({
            "reason_for_call": "Customer wanted to cancel and switch to Apex Mobile due to high pricing ($89/month).",
            "resolution": "Offered loyalty rate of $62/month locked for 12 months on Unlimited Plus plan. Customer accepted and stayed.",
            "resolution_status": "resolved",
            "follow_up_actions": ["Send confirmation email with new rate details"],
            "customer_sentiment": "positive",
            "key_entities": {"account_id": "AC-5567823", "product_mentioned": "Unlimited Plus plan", "dollar_amount": "$62"},
        })
    elif "AC-9912347" in prompt:
        # Intentionally includes a hallucination for Part 3 testing
        return json.dumps({
            "reason_for_call": "Customer needed an address update for an upcoming move and had an unrecognized charge.",
            "resolution": "Address updated. Internet transfer scheduled for next Tuesday. Device Protection Plus removed with $14.99 refunded. Dispute filed for remaining $29.98.",
            "resolution_status": "pending",
            "follow_up_actions": [
                "Installations team to schedule internet transfer within 48 hours",
                "Billing review to process $29.98 dispute within 5-7 business days",
                "Send customer a welcome package for the new address",
            ],
            "customer_sentiment": "neutral",
            "key_entities": {"account_id": "AC-9912347", "product_mentioned": "Device Protection Plus", "dollar_amount": "$14.99"},
        })
    elif "AC-2210038" in prompt:
        return json.dumps({
            "reason_for_call": "Customer asked about their contract end date.",
            "resolution": "Confirmed contract ends August 15th, 2026, then converts to month-to-month.",
            "resolution_status": "resolved",
            "follow_up_actions": [],
            "customer_sentiment": "positive",
            "key_entities": {"account_id": "AC-2210038", "product_mentioned": None, "dollar_amount": None},
        })
    elif "AC-6681200" in prompt:
        # Intentionally includes hallucinations for Part 3 testing
        return json.dumps({
            "reason_for_call": "Customer called for the third time about recurring billing overcharges. Charged $93 instead of promised $75 due to international calling add-on reattaching.",
            "resolution": "Agent identified a system bug causing the add-on to reattach. Transferred to supervisor Rachel. Agent also offered a 10% loyalty discount as compensation.",
            "resolution_status": "escalated",
            "follow_up_actions": [
                "Supervisor Rachel to fix system bug and authorize permanent add-on removal",
                "Process $54 refund for three months of overcharges",
                "Schedule a follow-up call in one week to verify the fix",
            ],
            "customer_sentiment": "angry",
            "key_entities": {"account_id": "AC-6681200", "product_mentioned": "international calling add-on", "dollar_amount": "$54"},
        })
    else:
        return json.dumps({
            "reason_for_call": "Unable to determine from transcript.",
            "resolution": "Unable to determine.",
            "resolution_status": "unresolved",
            "follow_up_actions": [],
            "customer_sentiment": "neutral",
            "key_entities": {"account_id": None, "product_mentioned": None, "dollar_amount": None},
        })


# ─────────────────────────────────────────────────────────────────────────────
# SCHEMA DEFINITION
# ─────────────────────────────────────────────────────────────────────────────

SUMMARY_SCHEMA = {
    "reason_for_call": {"type": "string", "required": True},
    "resolution": {"type": "string", "required": True},
    "resolution_status": {
        "type": "enum",
        "values": ["resolved", "escalated", "pending", "unresolved"],
        "required": True,
    },
    "follow_up_actions": {"type": "list_of_strings", "required": True},
    "customer_sentiment": {
        "type": "enum",
        "values": ["positive", "neutral", "frustrated", "angry"],
        "required": True,
    },
    "key_entities": {
        "type": "object",
        "required": True,
        "fields": {
            "account_id": {"type": "string_or_null"},
            "product_mentioned": {"type": "string_or_null"},
            "dollar_amount": {"type": "string_or_null"},
        },
    },
}


# ─────────────────────────────────────────────────────────────────────────────
# PART 1: TRANSCRIPT PARSING & PREPROCESSING                     [15 minutes]
# ─────────────────────────────────────────────────────────────────────────────
#
# Goals:
#   a) Compute transcript statistics (length, turn count, etc.)
#   b) Segment a transcript into phases (opening, problem, resolution, closing)
#   c) Extract key facts mentioned in the transcript for later verification
#
# Why this matters at Cresta:
#   Understanding transcript structure helps you build better prompts and
#   detect when summaries are hallucinating facts not in the source.
# ─────────────────────────────────────────────────────────────────────────────

def compute_transcript_stats(transcript: Transcript) -> Dict:
    """
    Return statistics about the transcript:
    {
        "total_turns": int,
        "agent_turns": int,
        "customer_turns": int,
        "total_words": int,
        "agent_words": int,
        "customer_words": int,
        "duration_seconds": int,
        "avg_words_per_turn": float,
        "talk_ratio": float,  # customer_words / agent_words
    }
    """
    # TODO: Implement this
    raise NotImplementedError("Part 1a: Implement compute_transcript_stats")


def segment_transcript(transcript: Transcript) -> Dict[str, List[Utterance]]:
    """
    Segment the transcript into conversation phases.

    Returns:
    {
        "opening": [...],       # greeting, identification (first 2-3 turns)
        "problem_statement": [...],  # customer explains their issue
        "investigation": [...],  # agent looks into the issue
        "resolution": [...],     # resolution is discussed/offered
        "closing": [...],        # wrap-up, goodbye (last 2-3 turns)
    }

    HINT: Use simple heuristics — this doesn't need to be ML-based.
    Think about: position in transcript, keywords, speaker patterns.

    Why this matters: Cresta's AI Summaries use phase-aware extraction to
    ensure no part of the conversation is missed. A summary that only captures
    the opening misses the resolution.
    """
    # TODO: Implement this
    raise NotImplementedError("Part 1b: Implement segment_transcript")


def extract_verifiable_facts(transcript: Transcript) -> List[Dict]:
    """
    Extract specific, verifiable facts from the transcript that MUST appear
    in any faithful summary. These become your hallucination checklist.

    Returns a list of dicts:
    [
        {"fact_type": "account_id", "value": "AC-XXXXXXX", "source_turn": int},
        {"fact_type": "dollar_amount", "value": "$XX.XX", "source_turn": int},
        {"fact_type": "product", "value": "...", "source_turn": int},
        {"fact_type": "action_promised", "value": "...", "source_turn": int},
        {"fact_type": "person_name", "value": "...", "source_turn": int},
        ...
    ]

    HINT: Use regex patterns for account IDs (AC-XXXXXXX), dollar amounts
    ($XX.XX), ticket numbers (TK-XXXXX), reference numbers (RF-XXXXX), etc.
    For action promises, look for agent utterances containing "I will",
    "I'm going to", "we'll", "you'll receive", etc.
    """
    # TODO: Implement this
    raise NotImplementedError("Part 1c: Implement extract_verifiable_facts")


# ─────────────────────────────────────────────────────────────────────────────
# PART 2: PROMPT ENGINEERING FOR STRUCTURED EXTRACTION            [25 minutes]
# ─────────────────────────────────────────────────────────────────────────────
#
# Build the prompt that drives summarization. This is the core of the system.
#
# Why this matters at Cresta:
#   Cresta's summaries are "fully customizable" — different clients want
#   different schema fields. The prompt must be robust to transcript variation
#   while always producing valid structured output.
# ─────────────────────────────────────────────────────────────────────────────

def build_system_prompt() -> str:
    """
    Write the system prompt for the summarization LLM.

    Requirements:
      - Define the assistant's role clearly
      - Specify the exact JSON output schema with field descriptions
      - Include constraints: faithfulness, no hallucination, brevity
      - Handle edge cases: what if the issue wasn't resolved? What if there
        are no follow-up actions? What if entities are missing?
      - Specify the enum values for resolution_status and customer_sentiment

    A strong system prompt is 200-400 words. Too short = ambiguous output.
    Too long = wasted tokens + the model ignores parts of it.
    """
    # TODO: Implement this — return the system prompt as a string
    raise NotImplementedError("Part 2a: Implement build_system_prompt")


def build_user_prompt(transcript: Transcript) -> str:
    """
    Build the user prompt containing the transcript and extraction request.

    Requirements:
      - Include the transcript in a clear, parseable format
      - Include metadata if available (team, call_id)
      - Remind the model to output ONLY valid JSON (no markdown fences,
        no explanatory text before/after)
      - Optionally include a one-shot example of the expected output format

    THINK ABOUT:
      - How do you format the transcript? Raw text? With timestamps?
        With speaker labels? Does the format affect extraction quality?
      - Should you include all utterances or summarize long transcripts?
      - Token budget: at serving time, you're paying per token. A 15-turn
        transcript is ~400 tokens. The prompt overhead should be minimal.
    """
    # TODO: Implement this — return the user prompt as a string
    raise NotImplementedError("Part 2b: Implement build_user_prompt")


def parse_llm_response(raw_response: str) -> Tuple[Optional[Dict], List[str]]:
    """
    Parse the LLM's raw text response into a validated structured summary.

    Returns:
      (parsed_summary, validation_errors)

    Where:
      - parsed_summary is the dict if valid, None if unparseable
      - validation_errors is a list of issues found, e.g.:
        ["resolution_status 'partial' is not a valid enum value",
         "key_entities.account_id is missing",
         "follow_up_actions should be a list, got str"]

    Requirements:
      - Handle markdown code fences (```json ... ```) that LLMs often add
      - Handle trailing commas, missing quotes, and other common JSON issues
      - Validate every field against SUMMARY_SCHEMA
      - Type-check: enums must match allowed values, lists must be lists, etc.
      - Return partial results when possible (don't throw away good fields
        because one field is invalid)

    Why this matters at Cresta:
      In production, ~5-10% of LLM outputs have formatting issues. A robust
      parser that recovers gracefully is worth more than a perfect prompt.
    """
    # TODO: Implement this
    raise NotImplementedError("Part 2c: Implement parse_llm_response")


def summarize_transcript(transcript: Transcript) -> Tuple[Dict, List[str]]:
    """
    End-to-end summarization pipeline:
      1. Build system + user prompts
      2. Call the LLM (use mock_llm_call)
      3. Parse and validate the response
      4. Return (summary_dict, validation_errors)

    This function ties Parts 2a-2c together.
    """
    # TODO: Implement this
    raise NotImplementedError("Part 2d: Implement summarize_transcript")


# ─────────────────────────────────────────────────────────────────────────────
# PART 3: FAITHFULNESS EVALUATION                                 [25 minutes]
# ─────────────────────────────────────────────────────────────────────────────
#
# A summary that sounds good but contains information NOT in the transcript
# is WORSE than no summary — it misleads the agent and can cause compliance
# violations. Build a hallucination detection system.
#
# Why this matters at Cresta:
#   Cresta serves airlines, banks, and insurers. A hallucinated dollar amount
#   in a summary could trigger a wrong refund. A fabricated promise could
#   create a legal obligation. Faithfulness is non-negotiable.
# ─────────────────────────────────────────────────────────────────────────────

def check_entity_faithfulness(
    summary: Dict,
    transcript: Transcript,
) -> List[Dict]:
    """
    Check whether entities in the summary (account IDs, dollar amounts,
    product names, etc.) are actually present in the transcript.

    Returns a list of issues found:
    [
        {
            "field": "key_entities.dollar_amount",
            "summary_value": "$54",
            "found_in_transcript": True/False,
            "severity": "critical",  # "critical" for $ amounts, "warning" for names
        },
        ...
    ]

    CRITICAL CHECK:
      - Dollar amounts in the summary MUST appear in the transcript
      - Account IDs MUST match exactly
      - Product names should be close matches (fuzzy OK)
    """
    # TODO: Implement this
    raise NotImplementedError("Part 3a: Implement check_entity_faithfulness")


def check_claim_faithfulness(
    summary: Dict,
    transcript: Transcript,
) -> List[Dict]:
    """
    Check whether claims in the summary's free-text fields (reason_for_call,
    resolution, follow_up_actions) are grounded in the transcript.

    For each claim, check:
      1. Key facts mentioned are actually in the transcript
      2. No actions are attributed that the agent didn't actually promise
      3. The resolution status matches what actually happened

    Returns:
    [
        {
            "field": "resolution",
            "claim": "Agent offered a 10% loyalty discount",
            "grounded": False,
            "evidence": "No mention of 10% loyalty discount in transcript",
            "severity": "critical",
        },
        {
            "field": "follow_up_actions",
            "claim": "Schedule a follow-up call in one week",
            "grounded": False,
            "evidence": "Agent did not promise any follow-up call",
            "severity": "warning",
        },
    ]

    HINT: This is hard to do perfectly without an LLM. For this exercise,
    implement a heuristic approach:
      - Extract key noun phrases from each summary field
      - Check if those phrases (or close variants) appear in the transcript
      - Flag any action verb phrases in follow_up_actions that don't have
        a corresponding agent promise in the transcript

    At Cresta, this would be done with a secondary "judge" LLM call, but
    the heuristic version is what they'd want to see you prototype first.
    """
    # TODO: Implement this
    raise NotImplementedError("Part 3b: Implement check_claim_faithfulness")


def faithfulness_score(
    summary: Dict,
    transcript: Transcript,
) -> Dict:
    """
    Compute an overall faithfulness score for a summary.

    Returns:
    {
        "score": float,          # 0.0 (all hallucinated) to 1.0 (fully faithful)
        "entity_issues": [...],  # from check_entity_faithfulness
        "claim_issues": [...],   # from check_claim_faithfulness
        "critical_count": int,   # number of critical-severity issues
        "warning_count": int,
        "pass": bool,            # True if critical_count == 0
    }

    The score should weight critical issues heavily:
      - Each critical issue: -0.25
      - Each warning: -0.10
      - Minimum score: 0.0
    """
    # TODO: Implement this
    raise NotImplementedError("Part 3c: Implement faithfulness_score")


# ─────────────────────────────────────────────────────────────────────────────
# PART 4: EVALUATION FRAMEWORK + PRODUCTION DESIGN                [25 minutes]
# ─────────────────────────────────────────────────────────────────────────────
#
# Build an evaluation framework that works WITHOUT ground-truth summaries
# (because in production, you don't have them).
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_summary_quality(
    summary: Dict,
    transcript: Transcript,
    ground_truth: Optional[Dict] = None,
) -> Dict:
    """
    Evaluate a single summary on multiple quality dimensions.

    Returns:
    {
        "faithfulness": float,     # from faithfulness_score()
        "completeness": float,     # are all major topics from the transcript covered?
        "conciseness": float,      # is the summary appropriately brief?
        "schema_validity": float,  # does the output match the expected schema?
        "overall_score": float,    # weighted combination

        # Optional (only if ground_truth provided):
        "ground_truth_similarity": float,  # how close to the reference summary?
    }

    DIMENSION DETAILS:

    Completeness (no ground truth needed):
      - Extract the conversation phases from segment_transcript()
      - Check: does the summary mention the problem? The resolution?
        The follow-up actions? Score based on coverage.

    Conciseness:
      - reason_for_call should be 1-2 sentences (15-40 words)
      - resolution should be 1-3 sentences (15-60 words)
      - Penalize if much longer or much shorter than expected

    Schema validity:
      - All required fields present?
      - Enum fields have valid values?
      - Types are correct?

    If ground_truth is provided:
      - Compute word-overlap similarity (simplified ROUGE-1) between
        generated and reference summaries for reason_for_call and resolution
    """
    # TODO: Implement this
    raise NotImplementedError("Part 4a: Implement evaluate_summary_quality")


def run_evaluation_suite(
    transcripts_and_truths: List[Tuple[Transcript, Dict]],
) -> Dict:
    """
    Run the full evaluation pipeline on all transcripts.

    Returns:
    {
        "num_transcripts": int,
        "per_call_results": [
            {"call_id": str, "scores": {...}, "issues": [...]}
        ],
        "aggregate": {
            "avg_faithfulness": float,
            "avg_completeness": float,
            "avg_conciseness": float,
            "avg_schema_validity": float,
            "avg_overall": float,
            "pass_rate": float,  # % of summaries with 0 critical issues
        },
    }
    """
    # TODO: Implement this
    raise NotImplementedError("Part 4b: Implement run_evaluation_suite")


def production_design() -> Dict:
    """
    No code needed — return a dict answering these design questions.
    """
    return {
        "latency_strategy": "",
        # The summary must be generated within 3 seconds of call end.
        # The transcript can be 50+ turns (~2000 tokens input).
        # How do you architect this for speed?
        # Consider: streaming, pre-processing during the call, model choice.

        "handling_long_transcripts": "",
        # Some calls last 30+ minutes with 100+ turns (~5000 tokens).
        # Your LLM context window is 4096 tokens. What's your strategy?
        # Compare: truncation, chunked summarization (map-reduce),
        # phase-based extraction, or using a longer-context model.

        "customization_strategy": "",
        # Different Cresta clients want different summary schemas.
        # An airline wants: {booking_ref, flight_number, issue_type, ...}
        # A bank wants: {account_number, transaction_id, complaint_type, ...}
        # How do you make the system configurable without rewriting prompts
        # for every client? Think: schema-driven prompt generation.

        "quality_monitoring": "",
        # In production, you have no ground truth. How do you monitor
        # summary quality continuously? How do you detect degradation?
        # Think: automated checks, human sampling strategy, feedback loops.

        "failure_modes": "",
        # Name the 3 most dangerous failure modes for a summarization
        # system in a contact center. For each, describe the business
        # impact and your mitigation strategy.
    }


# ─────────────────────────────────────────────────────────────────────────────
# RUNNER
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print("=" * 70)
    print("CRESTA MLE INTERVIEW — Call Transcript Summarization")
    print("=" * 70)

    data = get_all_transcripts()
    print(f"\nLoaded {len(data)} transcripts\n")

    # ── Part 1 ──────────────────────────────────────────────────────────
    print("📊 PART 1: Transcript Parsing & Preprocessing")
    print("-" * 50)

    transcript, gt = data[0]  # Use the billing dispute for Part 1 demos

    try:
        stats = compute_transcript_stats(transcript)
        print(f"  Call {transcript.call_id}:")
        print(f"    Turns: {stats['total_turns']} (agent: {stats['agent_turns']}, customer: {stats['customer_turns']})")
        print(f"    Words: {stats['total_words']} (agent: {stats['agent_words']}, customer: {stats['customer_words']})")
        print(f"    Duration: {stats['duration_seconds']}s | Avg words/turn: {stats['avg_words_per_turn']:.1f}")
        print(f"    Talk ratio (cust/agent): {stats['talk_ratio']:.2f}")
        print("  ✅ Part 1a passed")
    except NotImplementedError:
        print("  ⬜ Part 1a: Not implemented yet")

    try:
        phases = segment_transcript(transcript)
        for phase, utts in phases.items():
            if utts:
                preview = utts[0].text[:60] + "..." if len(utts[0].text) > 60 else utts[0].text
                print(f"    {phase:<22s} {len(utts)} turns — \"{preview}\"")
        print("  ✅ Part 1b passed")
    except NotImplementedError:
        print("  ⬜ Part 1b: Not implemented yet")

    try:
        facts = extract_verifiable_facts(transcript)
        print(f"  Extracted {len(facts)} verifiable facts:")
        for f in facts[:5]:
            print(f"    [{f['fact_type']}] {f['value']} (turn {f['source_turn']})")
        print("  ✅ Part 1c passed")
    except NotImplementedError:
        print("  ⬜ Part 1c: Not implemented yet")

    # ── Part 2 ──────────────────────────────────────────────────────────
    print(f"\n🤖 PART 2: Prompt Engineering & Summarization")
    print("-" * 50)

    try:
        sys_prompt = build_system_prompt()
        assert isinstance(sys_prompt, str) and len(sys_prompt) > 50
        print(f"  System prompt: {len(sys_prompt)} chars, {len(sys_prompt.split())} words")
        print(f"  Preview: \"{sys_prompt[:100]}...\"")
        print("  ✅ Part 2a passed")
    except NotImplementedError:
        print("  ⬜ Part 2a: Not implemented yet")

    try:
        user_prompt = build_user_prompt(transcript)
        assert isinstance(user_prompt, str)
        print(f"  User prompt: {len(user_prompt)} chars")
        print("  ✅ Part 2b passed")
    except NotImplementedError:
        print("  ⬜ Part 2b: Not implemented yet")

    try:
        raw = '```json\n{"reason_for_call": "test", "extra_newline": "a\\nb"}\n```'
        parsed, errors = parse_llm_response(raw)
        assert parsed is not None, "Parser should handle markdown fences"
        print(f"  Parser test (markdown fences): OK — {len(errors)} validation notes")
        print("  ✅ Part 2c passed")
    except NotImplementedError:
        print("  ⬜ Part 2c: Not implemented yet")
    except AssertionError as e:
        print(f"  ❌ Part 2c: {e}")

    try:
        summary, errors = summarize_transcript(transcript)
        if summary:
            print(f"  Summary for {transcript.call_id}:")
            print(f"    Reason: {summary.get('reason_for_call', 'N/A')[:80]}...")
            print(f"    Status: {summary.get('resolution_status', 'N/A')}")
            print(f"    Sentiment: {summary.get('customer_sentiment', 'N/A')}")
            print(f"    Validation errors: {len(errors)}")
        print("  ✅ Part 2d passed")
    except NotImplementedError:
        print("  ⬜ Part 2d: Not implemented yet")
    except Exception as e:
        print(f"  ❌ Part 2d error: {e}")

    # ── Part 3 ──────────────────────────────────────────────────────────
    print(f"\n🔍 PART 3: Faithfulness Evaluation")
    print("-" * 50)

    # Use the transcripts that have intentional hallucinations in mock output
    test_cases = [data[3], data[5]]  # complex multi-issue + angry unresolved

    for transcript, gt in test_cases:
        try:
            summary, _ = summarize_transcript(transcript)
            if summary is None:
                print(f"  ⬜ Part 3: Need Part 2 working first")
                break

            score_result = faithfulness_score(summary, transcript)
            status = "✅ PASS" if score_result["pass"] else "❌ FAIL"
            print(f"  Call {transcript.call_id}: faithfulness={score_result['score']:.2f} "
                  f"critical={score_result['critical_count']} "
                  f"warnings={score_result['warning_count']} {status}")
            for issue in score_result["entity_issues"] + score_result["claim_issues"]:
                if not issue.get("grounded", issue.get("found_in_transcript", True)):
                    severity = issue.get("severity", "info")
                    desc = issue.get("claim", issue.get("summary_value", "?"))
                    print(f"    ⚠ [{severity}] {issue.get('field', '?')}: \"{desc[:60]}\"")

        except NotImplementedError:
            print(f"  ⬜ Part 3: Not implemented yet")
            break
        except Exception as e:
            print(f"  ❌ Part 3 error on {transcript.call_id}: {e}")

    # ── Part 4 ──────────────────────────────────────────────────────────
    print(f"\n📋 PART 4: Evaluation Framework")
    print("-" * 50)

    try:
        suite_result = run_evaluation_suite(data)
        agg = suite_result["aggregate"]
        print(f"  Evaluated {suite_result['num_transcripts']} transcripts")
        print(f"  Avg faithfulness:    {agg['avg_faithfulness']:.3f}")
        print(f"  Avg completeness:    {agg['avg_completeness']:.3f}")
        print(f"  Avg conciseness:     {agg['avg_conciseness']:.3f}")
        print(f"  Avg schema validity: {agg['avg_schema_validity']:.3f}")
        print(f"  Avg overall:         {agg['avg_overall']:.3f}")
        print(f"  Pass rate:           {agg['pass_rate']:.1%}")
        print("  ✅ Part 4a+b passed")
    except NotImplementedError:
        print("  ⬜ Part 4a/b: Not implemented yet")
    except Exception as e:
        print(f"  ❌ Part 4 error: {e}")

    try:
        design = production_design()
        filled = sum(1 for v in design.values() if v.strip())
        total = len(design)
        print(f"\n  Production design: {filled}/{total} questions answered")
        if filled == total:
            print("  ✅ Part 4c passed")
        else:
            missing = [k for k, v in design.items() if not v.strip()]
            print(f"  ⬜ Missing: {', '.join(missing)}")
    except Exception as e:
        print(f"  ❌ Part 4c error: {e}")

    print("\n" + "=" * 70)
    print("DONE — Review output above for pass/fail on each part.")
    print("=" * 70)


if __name__ == "__main__":
    main()

In [ ]:
"""
================================================================================
SIMPLE SOLUTION — Cresta Summarization (realistic for a 1-hour interview)
================================================================================
Focus: clean code, correct logic, good enough heuristics. No over-engineering.
================================================================================
"""

import json
import re
from typing import Dict, List, Optional, Tuple
from collections import Counter

from cresta_summarization_problem import (
    Transcript,
    Utterance,
    SUMMARY_SCHEMA,
    get_all_transcripts,
    mock_llm_call,
)


# ═══════════════════════════════════════════════════════════════════════
# PART 1 — just count things and use simple splits
# ═══════════════════════════════════════════════════════════════════════

def compute_transcript_stats(transcript: Transcript) -> Dict:
    agent_utts = [u for u in transcript.utterances if u.role == "agent"]
    cust_utts = [u for u in transcript.utterances if u.role == "customer"]
    agent_words = sum(len(u.text.split()) for u in agent_utts)
    cust_words = sum(len(u.text.split()) for u in cust_utts)

    return {
        "total_turns": len(transcript.utterances),
        "agent_turns": len(agent_utts),
        "customer_turns": len(cust_utts),
        "total_words": agent_words + cust_words,
        "agent_words": agent_words,
        "customer_words": cust_words,
        "duration_seconds": transcript.total_duration_sec(),
        "avg_words_per_turn": (agent_words + cust_words) / max(len(transcript.utterances), 1),
        "talk_ratio": cust_words / max(agent_words, 1),
    }


def segment_transcript(transcript: Transcript) -> Dict[str, List[Utterance]]:
    """Simple approach: opening = first 2, closing = last 2, split the rest."""
    utts = transcript.utterances
    n = len(utts)

    if n <= 4:
        return {
            "opening": utts[:1],
            "problem_statement": utts[1:2],
            "investigation": utts[2:3],
            "resolution": utts[3:4] if n > 3 else [],
            "closing": utts[-1:] if n > 1 else [],
        }

    opening = utts[:2]
    closing = utts[-2:]
    middle = utts[2:-2]

    # Simple split: first third = problem, middle third = investigation, last third = resolution
    third = max(1, len(middle) // 3)

    return {
        "opening": opening,
        "problem_statement": middle[:third],
        "investigation": middle[third:third * 2],
        "resolution": middle[third * 2:],
        "closing": closing,
    }


def extract_verifiable_facts(transcript: Transcript) -> List[Dict]:
    """Regex for structured entities. Keep it simple."""
    facts = []

    for i, u in enumerate(transcript.utterances):
        # Account IDs
        for m in re.finditer(r'AC-\d+', u.text):
            facts.append({"fact_type": "account_id", "value": m.group(), "source_turn": i})

        # Dollar amounts
        for m in re.finditer(r'\$[\d.]+', u.text):
            facts.append({"fact_type": "dollar_amount", "value": m.group(), "source_turn": i})

        # Ticket / reference numbers
        for m in re.finditer(r'[A-Z]{2}-\d{4,}', u.text):
            facts.append({"fact_type": "reference", "value": m.group(), "source_turn": i})

    # Deduplicate
    seen = set()
    unique = []
    for f in facts:
        key = (f["fact_type"], f["value"])
        if key not in seen:
            seen.add(key)
            unique.append(f)
    return unique


# ═══════════════════════════════════════════════════════════════════════
# PART 2 — prompt + parse + summarize
# ═══════════════════════════════════════════════════════════════════════

def build_system_prompt() -> str:
    return """You are a contact center call summarizer. Given a transcript, output a JSON object with these exact fields:

{
  "reason_for_call": "1-2 sentences, why the customer called",
  "resolution": "1-3 sentences, what the agent did to help",
  "resolution_status": one of "resolved" | "escalated" | "pending" | "unresolved",
  "follow_up_actions": ["list of promised next steps, or empty array"],
  "customer_sentiment": one of "positive" | "neutral" | "frustrated" | "angry",
  "key_entities": {
    "account_id": "string or null",
    "product_mentioned": "string or null",
    "dollar_amount": "string or null"
  }
}

Rules:
- Only include facts explicitly stated in the transcript.
- Do not invent or assume anything.
- Output valid JSON only, no markdown."""


def build_user_prompt(transcript: Transcript) -> str:
    return f"""Summarize this call transcript:

{transcript.to_text()}

Output JSON only."""


def parse_llm_response(raw_response: str) -> Tuple[Optional[Dict], List[str]]:
    errors = []
    text = raw_response.strip()

    # Strip markdown fences
    text = re.sub(r'^```(?:json)?\s*', '', text)
    text = re.sub(r'\s*```$', '', text)
    text = text.strip()

    # Parse
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError as e:
        # One recovery attempt: fix trailing commas
        fixed = re.sub(r',\s*([}\]])', r'\1', text)
        try:
            parsed = json.loads(fixed)
        except json.JSONDecodeError:
            return None, [f"JSON parse error: {e}"]

    if not isinstance(parsed, dict):
        return None, ["Response is not a JSON object"]

    # Basic validation
    required = ["reason_for_call", "resolution", "resolution_status",
                 "follow_up_actions", "customer_sentiment", "key_entities"]
    for field in required:
        if field not in parsed:
            errors.append(f"Missing field: {field}")

    valid_statuses = {"resolved", "escalated", "pending", "unresolved"}
    if parsed.get("resolution_status") not in valid_statuses:
        errors.append(f"Invalid resolution_status: {parsed.get('resolution_status')}")

    valid_sentiments = {"positive", "neutral", "frustrated", "angry"}
    if parsed.get("customer_sentiment") not in valid_sentiments:
        errors.append(f"Invalid customer_sentiment: {parsed.get('customer_sentiment')}")

    if not isinstance(parsed.get("follow_up_actions"), list):
        errors.append("follow_up_actions should be a list")

    return parsed, errors


def summarize_transcript(transcript: Transcript) -> Tuple[Dict, List[str]]:
    sys_prompt = build_system_prompt()
    user_prompt = build_user_prompt(transcript)
    raw = mock_llm_call(user_prompt, sys_prompt)
    return parse_llm_response(raw)


# ═══════════════════════════════════════════════════════════════════════
# PART 3 — hallucination detection (keep it simple)
# ═══════════════════════════════════════════════════════════════════════

def check_entity_faithfulness(summary: Dict, transcript: Transcript) -> List[Dict]:
    """Check if entities in the summary actually appear in the transcript."""
    issues = []
    raw_text = transcript.to_text()

    entities = summary.get("key_entities", {})

    # Account ID — exact match
    acct = entities.get("account_id")
    if acct:
        issues.append({
            "field": "key_entities.account_id",
            "summary_value": acct,
            "found_in_transcript": acct in raw_text,
            "severity": "critical",
        })

    # Dollar amount — exact match
    amount = entities.get("dollar_amount")
    if amount:
        issues.append({
            "field": "key_entities.dollar_amount",
            "summary_value": amount,
            "found_in_transcript": amount in raw_text,
            "severity": "critical",
        })

    # Also catch dollar amounts hiding in free-text fields
    for field in ["reason_for_call", "resolution"]:
        for amt in re.findall(r'\$[\d.]+', summary.get(field, "")):
            if amt not in raw_text:
                issues.append({
                    "field": field,
                    "summary_value": amt,
                    "found_in_transcript": False,
                    "severity": "critical",
                })

    return issues


def check_claim_faithfulness(summary: Dict, transcript: Transcript) -> List[Dict]:
    """
    Simple approach: for each follow_up_action, check if enough of its
    significant words appear in the transcript. If not, flag it.
    """
    issues = []
    transcript_lower = transcript.to_text().lower()
    stopwords = {"the", "a", "an", "to", "for", "and", "or", "in", "on", "of",
                 "is", "be", "will", "should", "would", "with", "from", "that",
                 "this", "their", "them", "they", "have", "has", "been", "within"}

    for action in summary.get("follow_up_actions", []):
        words = set(action.lower().split()) - stopwords
        words = {w for w in words if len(w) >= 4}  # only meaningful words

        if not words:
            continue

        hits = sum(1 for w in words if w in transcript_lower)
        overlap = hits / len(words)

        if overlap < 0.5:
            issues.append({
                "field": "follow_up_actions",
                "claim": action,
                "grounded": False,
                "evidence": f"Only {hits}/{len(words)} key words found in transcript",
                "severity": "warning",
            })

    return issues


def faithfulness_score(summary: Dict, transcript: Transcript) -> Dict:
    entity_issues = check_entity_faithfulness(summary, transcript)
    claim_issues = check_claim_faithfulness(summary, transcript)

    critical = sum(1 for i in entity_issues if not i.get("found_in_transcript", True) and i["severity"] == "critical")
    warnings = sum(1 for i in claim_issues if not i.get("grounded", True))

    score = max(0.0, 1.0 - critical * 0.25 - warnings * 0.10)

    return {
        "score": round(score, 2),
        "entity_issues": entity_issues,
        "claim_issues": claim_issues,
        "critical_count": critical,
        "warning_count": warnings,
        "pass": critical == 0,
    }


# ═══════════════════════════════════════════════════════════════════════
# PART 4 — evaluation
# ═══════════════════════════════════════════════════════════════════════

def evaluate_summary_quality(
    summary: Dict,
    transcript: Transcript,
    ground_truth: Optional[Dict] = None,
) -> Dict:
    # Faithfulness
    faith = faithfulness_score(summary, transcript)

    # Completeness — did we capture the key entities?
    facts = extract_verifiable_facts(transcript)
    summary_text = json.dumps(summary).lower()
    found = sum(1 for f in facts if f["value"].lower() in summary_text)
    completeness = found / max(len(facts), 1)

    # Conciseness — penalize if too long or too short
    reason_len = len(summary.get("reason_for_call", "").split())
    res_len = len(summary.get("resolution", "").split())
    conciseness = 1.0
    if reason_len < 5 or reason_len > 60:
        conciseness -= 0.25
    if res_len < 5 or res_len > 100:
        conciseness -= 0.25

    # Schema validity
    _, errors = parse_llm_response(json.dumps(summary))
    schema_validity = 1.0 if len(errors) == 0 else max(0.0, 1.0 - len(errors) * 0.2)

    overall = (
        faith["score"] * 0.40 +
        completeness * 0.25 +
        conciseness * 0.15 +
        schema_validity * 0.20
    )

    result = {
        "faithfulness": faith["score"],
        "completeness": round(completeness, 3),
        "conciseness": round(conciseness, 3),
        "schema_validity": round(schema_validity, 3),
        "overall_score": round(overall, 3),
    }

    # Optional: ground truth word overlap (simplified ROUGE-1)
    if ground_truth:
        scores = []
        for field in ["reason_for_call", "resolution"]:
            gen = set(summary.get(field, "").lower().split())
            ref = set(ground_truth.get(field, "").lower().split())
            if gen and ref:
                overlap = gen & ref
                p = len(overlap) / len(gen)
                r = len(overlap) / len(ref)
                f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
                scores.append(f1)
        if scores:
            result["ground_truth_similarity"] = round(sum(scores) / len(scores), 3)

    return result


def run_evaluation_suite(transcripts_and_truths):
    results = []
    for transcript, gt in transcripts_and_truths:
        summary, errors = summarize_transcript(transcript)
        if summary is None:
            results.append({"call_id": transcript.call_id, "scores": {"overall_score": 0}, "issues": errors})
            continue

        scores = evaluate_summary_quality(summary, transcript, gt)
        faith = faithfulness_score(summary, transcript)
        issues = [i for i in faith["entity_issues"] if not i.get("found_in_transcript", True)]
        issues += [i for i in faith["claim_issues"] if not i.get("grounded", True)]
        results.append({"call_id": transcript.call_id, "scores": scores, "issues": issues})

    def avg(key):
        vals = [r["scores"].get(key, 0) for r in results]
        return round(sum(vals) / max(len(vals), 1), 3)

    return {
        "num_transcripts": len(results),
        "per_call_results": results,
        "aggregate": {
            "avg_faithfulness": avg("faithfulness"),
            "avg_completeness": avg("completeness"),
            "avg_conciseness": avg("conciseness"),
            "avg_schema_validity": avg("schema_validity"),
            "avg_overall": avg("overall_score"),
            "pass_rate": round(sum(1 for r in results if not r["issues"]) / max(len(results), 1), 3),
        },
    }


def production_design() -> Dict:
    return {
        "latency_strategy": (
            "Use a small fast model (Haiku/GPT-4o-mini) since structured extraction "
            "doesn't need the biggest model. Pre-build the prompt during the call so "
            "at call-end you just append the last few turns and fire. Stream the response."
        ),
        "handling_long_transcripts": (
            "For calls under 4K tokens, send the full transcript. For longer calls, "
            "keep the first 2 turns (context) + last 60% of the call (resolution is "
            "usually at the end). Truncating the middle is safer than truncating the end."
        ),
        "customization_strategy": (
            "Make the system prompt template-driven: each client provides a JSON schema "
            "definition, and the prompt is generated from it. Adding a new client = "
            "adding a config file, not changing code."
        ),
        "quality_monitoring": (
            "Track agent accept/edit rate on summaries. If agents edit a specific field "
            "more than 30% of the time, that field's prompt needs work. Sample 2% for "
            "human QA review weekly."
        ),
        "failure_modes": (
            "1) Hallucinated dollar amounts → wrong refund processed → entity checker blocks these. "
            "2) Fabricated promises → customer expects something never offered → check actions against transcript. "
            "3) Wrong resolution_status → ticket closed prematurely → use keyword rules as backup check."
        ),
    }


# ═══════════════════════════════════════════════════════════════════════
# RUNNER
# ═══════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print("=" * 60)
    print("SIMPLE SOLUTION — Summarization")
    print("=" * 60)

    data = get_all_transcripts()

    # Part 1
    t, gt = data[0]
    stats = compute_transcript_stats(t)
    print(f"\n📊 Part 1: {t.call_id} — {stats['total_turns']} turns, {stats['total_words']} words")

    facts = extract_verifiable_facts(t)
    print(f"   Extracted {len(facts)} facts: {[f['value'] for f in facts]}")

    # Part 2
    print(f"\n🤖 Part 2: Summarizing all calls")
    for t, gt in data:
        summary, errors = summarize_transcript(t)
        if summary:
            print(f"   {t.call_id}: status={summary['resolution_status']}, "
                  f"sentiment={summary['customer_sentiment']}, errors={len(errors)}")

    # Part 3
    print(f"\n🔍 Part 3: Faithfulness")
    for t, gt in data:
        summary, _ = summarize_transcript(t)
        if summary:
            faith = faithfulness_score(summary, t)
            status = "PASS" if faith["pass"] else "FAIL"
            print(f"   {t.call_id}: score={faith['score']:.2f} [{status}]", end="")
            if not faith["pass"]:
                bad = [i["summary_value"] for i in faith["entity_issues"]
                       if not i.get("found_in_transcript", True)]
                print(f" — hallucinated: {bad}", end="")
            print()

    # Part 4
    print(f"\n📋 Part 4: Evaluation Suite")
    result = run_evaluation_suite(data)
    agg = result["aggregate"]
    for k, v in agg.items():
        print(f"   {k}: {v}")

    print("\n" + "=" * 60)